# Normalizing Flows

Neste notebook, exploraremos **Normalizing Flows** (Fluxos Normalizadores), uma das famílias de modelos generativos baseados em verossimilhança (*likelihood-based models*). Aprofundaremos nosso conhecimento através de duas abordagens complementares:

1. **Entendimento Conceitual e Geométrico (2D):** Utilizaremos a distribuição de duas luas (*make_moons*) para visualizar fisicamente a deformação contínua do espaço bidimensional.
2. **Modelagem de Alta Dimensão (Imagens MNIST):** Escalaremos o mesmo conceito para imagens digitais de $28\times28$ pixels ($784$ dimensões).

Diferente das GANs (que não oferecem uma densidade explícita) e dos VAEs (que otimizam uma aproximação chamada ELBO), os Normalizing Flows nos permitem calcular a **verossimilhança exata** dos dados e realizam **inferência e geração exatas** por meio de transformações bijetoras.

## Fundamentos

### O Teorema da Mudança de Variáveis

A base dos Normalizing Flows reside no Teorema da Mudança de Variáveis. Se temos uma variável aleatória latente $Z$ com distribuição conhecida $p_Z(z)$ (a distribuição base, tipicamente uma Gaussiana Standard $\mathcal{N}(0, I)$) e aplicamos uma transformação bijetora (invertível) e diferenciável $x = f^{-1}(z)$ (ou $z = f(x)$), a densidade de probabilidade da variável observada $X$ é dada por:

$$
p_X(x) = p_Z(f(x)) \left| \det \left( \frac{\partial f(x)}{\partial x} \right) \right|
$$

Aplicando o logaritmo em ambos os lados, obtemos o log-likelihood:

$$
\log p_X(x) = \log p_Z(f(x)) + \log \left| \det \left( \frac{\partial f(x)}{\partial x} \right) \right|
$$

Onde a matriz Jacobiana $J = \frac{\partial f(x)}{\partial x}$ representa a taxa de variação local e o determinante $|\det J|$ mede a dilatação ou compressão de volume induzida pela transformação $f$.

### Camada de Acoplamento Afim (Affine Coupling Layer)

Para projetar transformações expressivas e com determinante computacionalmente barato, usamos as **Camadas de Acoplamento Afim** (RealNVP). Dividimos a entrada $x$ em duas partes ($x_1$ e $x_2$) usando uma máscara binária:

$$
y_1 = x_1
$$
$$
y_2 = x_2 \odot \exp(s(x_1)) + t(x_1)
$$

Onde $s$ (escala) e $t$ (translação) são redes neurais complexas arbitrárias (que não precisam ser invertíveis!).

#### Propriedades:
1. **Inversão Simples:**
$$
x_1 = y_1
$$
$$
x_2 = (y_2 - t(y_1)) \odot \exp(-s(y_1))
$$

2. **Jacobiano Triangular:** O determinante é apenas o produto da diagonal:
$$
\log |\det J| = \sum s(x_1)
$$

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.animation as animation
from IPython.display import Image, display

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Exemplo Conceitual

Nesta seção, trabalharemos com a distribuição bidimensional clássica **make_moons** do *scikit-learn*. Como os dados possuem apenas $2$ dimensões, podemos plotar a nuvem de pontos diretamente no plano cartesiano.

O objetivo é entender conceitualmente como o fluxo consegue deformar uma densidade circular uniforme (uma Gaussiana $2D$) para se adaptar ao formato de duas luas interlaçadas.

In [ ]:
# Gerar os dados das duas luas
X_np, _ = make_moons(n_samples=3000, noise=0.05, random_state=42)

# Normalizar os dados para terem média zero e variância unitária (estabiliza o treino)
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)

# Converter para tensores PyTorch
X_tensor = torch.tensor(X_np, dtype=torch.float32)
moons_dataset = TensorDataset(X_tensor)
moons_dataloader = DataLoader(moons_dataset, batch_size=128, shuffle=True)

# Plotar scatter plot inicial da distribuição real 2D
plt.figure(figsize=(6, 5))
plt.scatter(X_np[:, 0], X_np[:, 1], s=10, alpha=0.6, color='#8a2be2')
plt.title("Distribuição Real de Dados (Duas Luas)")
plt.xlabel("X1")
plt.ylabel("X2")
plt.grid(True, alpha=0.3)
plt.show()

### Arquitetura Unificada do Modelo

Definiremos a seguir as classes de PyTorch que servirão tanto para o exemplo 2D quanto para o exemplo de imagens MNIST:

- `AffineCouplingLayer`: Realiza a transformação afim de metade das variáveis mantendo a outra metade constante com base em uma máscara (`self.mask`). A rede interna `self.net` é uma MLP rápida.
- `NormalizingFlow`: Gerencia o empilhamento das camadas e a alternância automática de máscaras (padrão de xadrez para imagens MNIST e padrão `[1, 0]` alternado para o espaço 2D).

In [ ]:
class AffineCouplingLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim, mask):
        super().__init__()
        # Registra a máscara como buffer não treinável
        self.register_buffer('mask', mask)
        
        # MLP para escala (s) e translação (t)
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim * 2)
        )

    def forward(self, x):
        # Entrada x: batch_size x input_dim
        x_masked = x * self.mask
        out = self.net(x_masked)
        s, t = out.chunk(2, dim=1) # Divide em blocos de tamanho input_dim
        
        s = torch.tanh(s) # Estabilidade numérica para a escala
        
        # Transformação afim nas variáveis não mascaradas
        y = x_masked + (1 - self.mask) * (x * torch.exp(s) + t)
        
        # Log-determinante do Jacobiano triangular
        log_det_jacobian = ((1 - self.mask) * s).sum(dim=1)
        
        return y, log_det_jacobian
    
    def inverse(self, y):
        y_masked = y * self.mask
        out = self.net(y_masked)
        s, t = out.chunk(2, dim=1)
        
        s = torch.tanh(s)
        
        # Inversa da transformação afim
        x = y_masked + (1 - self.mask) * ((y - t) * torch.exp(-s))
        return x

In [ ]:
class NormalizingFlow(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        self.layers = nn.ModuleList()
        
        # Configuração da máscara de acordo com a dimensionalidade
        if input_dim == 2:
            mask = torch.tensor([1.0, 0.0])
        else:
            mask = torch.zeros(input_dim)
            mask[::2] = 1.0
            
        for i in range(num_layers):
            current_mask = mask if i % 2 == 0 else 1.0 - mask
            self.layers.append(AffineCouplingLayer(input_dim, hidden_dim, current_mask))
            
    def forward(self, x):
        # Mapeamento Direto: x (dados) -> z (latente)
        log_det_jacobian = 0
        for layer in self.layers:
            x, ldj = layer(x)
            log_det_jacobian += ldj
        return x, log_det_jacobian
    
    def forward_steps(self, x):
        # Retorna todos os passos intermediários do mapeamento x -> z
        steps = [x.cpu().numpy()]
        for layer in self.layers:
            x, _ = layer(x)
            steps.append(x.cpu().numpy())
        return steps
    
    def inverse(self, z):
        # Mapeamento Inverso: z (latente) -> x (dados)
        for layer in reversed(self.layers):
            z = layer.inverse(z)
        return z
    
    def inverse_steps(self, z):
        # Retorna todos os passos intermediários do mapeamento z -> x
        steps = [z.cpu().numpy()]
        for layer in reversed(self.layers):
            z = layer.inverse(z)
            steps.append(z.cpu().numpy())
        return steps

In [ ]:
input_dim_2d = 2
hidden_dim_2d = 128
num_layers_2d = 8

model_2d = NormalizingFlow(input_dim_2d, hidden_dim_2d, num_layers_2d).to(device)

### Treinamento e Evolução Visual das Épocas (make_moons)

Minimizaremos a **Negative Log-Likelihood (NLL)**. Durante as 20 épocas de treinamento, vamos gerar scatter plots intermediários em épocas específicas (`1, 5, 10, 15, 20`) para observar visualmente a transição contínua da Gaussiana Standard para a forma das duas luas.

In [ ]:
optimizer_2d = optim.Adam(model_2d.parameters(), lr=1e-3)
base_dist_2d = torch.distributions.Normal(torch.zeros(input_dim_2d).to(device), torch.ones(input_dim_2d).to(device))

In [ ]:
num_epochs_2d = 50
losses_2d = []
generated_frames = []
frame_epochs = []

def sample_model():
    model_2d.eval()
    with torch.no_grad():
        z = base_dist_2d.sample((1000,))
        return model_2d.inverse(z).cpu().numpy()

# Época 0
generated_frames.append(sample_model())
frame_epochs.append(0)

for epoch in range(1, num_epochs_2d + 1):
    model_2d.train()
    epoch_loss = 0

    for (real_x,) in moons_dataloader:
        real_x = real_x.to(device)

        optimizer_2d.zero_grad()

        # Mapeamento x -> z
        z, log_det = model_2d(real_x)

        # Log-likelihood dos dados reais
        log_prob_z = base_dist_2d.log_prob(z).sum(dim=1)
        loss = -(log_prob_z + log_det).mean()

        loss.backward()
        optimizer_2d.step()

        epoch_loss += loss.item()

    losses_2d.append(epoch_loss / len(moons_dataloader))

    generated_frames.append(sample_model())
    frame_epochs.append(epoch)

    print(f"[Época {epoch}/{num_epochs_2d}] Loss: {losses_2d[-1]:.4f}")

In [ ]:
epochs_to_plot = [0, 1, 5, 10, 15, 20, 50]

fig, axs = plt.subplots(
    1,
    len(epochs_to_plot),
    figsize=(3.8 * len(epochs_to_plot), 3.5)
)

for ax, epoch in zip(np.ravel(axs), epochs_to_plot):
    idx = frame_epochs.index(epoch)
    x_gen = generated_frames[idx]

    ax.scatter(x_gen[:, 0], x_gen[:, 1], s=4, alpha=0.6, color="#e91e63")
    ax.set_title(f"Época {epoch}")
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.grid(True, alpha=0.3)

plt.suptitle("Evolução das Amostras Geradas Durante o Treinamento", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

scat = ax.scatter([], [], s=4, alpha=0.6, color='#9c27b0')
title = ax.set_title("")

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.grid(True, alpha=0.3)

def init():
    scat.set_offsets(np.empty((0, 2)))
    title.set_text("")
    return scat, title

def update(frame_idx):
    generated_x = generated_frames[frame_idx]
    epoch = frame_epochs[frame_idx]
    
    scat.set_offsets(generated_x)
    title.set_text(f"Treinamento do Normalizing Flow - Época {epoch}")
    
    return scat, title

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(generated_frames),
    init_func=init,
    blit=True,
    interval=120
)

gif_path = "normalizing_flow_training_2d.gif"
ani.save(gif_path, writer='pillow', fps=10)

plt.close()

display(Image(filename=gif_path))

In [ ]:
model_2d.eval()

with torch.no_grad():
    z_samples = base_dist_2d.sample((50,))
    inv_steps = model_2d.inverse_steps(z_samples)

layer_frames = [
    step.detach().cpu().numpy() if torch.is_tensor(step) else step
    for step in inv_steps
]

num_layers = len(layer_frames)
cols = 5
rows = int(np.ceil(num_layers / cols))

fig, axs = plt.subplots(
    rows,
    cols,
    figsize=(3.5 * cols, 3.5 * rows)
)

axs = np.ravel(axs)

for i, ax in enumerate(axs):
    if i < num_layers:
        points = layer_frames[i]

        ax.scatter(
            points[:, 0],
            points[:, 1],
            s=20,
            alpha=0.6,
            color="#9c27b0"
        )

        if i == 0:
            ax.set_title("Base z")
        elif i == num_layers - 1:
            ax.set_title("Final x")
        else:
            ax.set_title(f"Layer {i}")

        ax.set_xlim(-3, 3)
        ax.set_ylim(-3, 3)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(True, alpha=0.3)
    else:
        ax.axis("off")

plt.suptitle(
    "Transformação camada a camada: z → x",
    fontsize=14,
    y=1.02
)

plt.tight_layout()
plt.show()

## Aplicação em Imagens (MNIST)

Agora que entendemos visualmente como os Normalizing Flows moldam o espaço de dados, vamos escalar o conceito para **imagens reais** do dataset **MNIST**.

Cada imagem digital possui dimensões de $28\times28$ pixels, o que significa que o nosso espaço de dados $X$ possui **784 dimensões** ($X \in \mathbb{R}^{784}$). Mapearemos essas imagens para um espaço latente Gaussiano de mesma dimensão ($Z \in \mathbb{R}^{784}$).

### O Conceito Crítico de Dequantization (Dequantização)

Os Normalizing Flows são modelos contínuos que estimam densidades de probabilidade. No entanto, as imagens digitais são inerentemente **discretas** (pixels possuem valores inteiros de $0$ a $255$, que após a divisão por $255$, pertencem ao conjunto discreto $\{0, 1/255, 2/255, \dots, 1\}$).

Se tentarmos treinar uma distribuição contínua diretamente em pontos discretos, o modelo colapsará rapidamente, gerando densidades de probabilidade infinitas (funções delta de Dirac) em cima desses valores exatos dos pixels (ocasionando *overfitting* extremo).

Para evitar esse problema, aplicamos a **Dequantização Uniforme**, adicionando ruído uniforme controlado a cada pixel durante o treinamento:

$$
x_{\text{continuo}} = x_{\text{discreto}} + u \quad \text{onde} \quad u \sim \mathcal{U}(0, 1/256)
$$

Isso transforma a distribuição discreta em uma distribuição contínua por partes, garantindo a estabilidade e a convergência matemática correta do fluxo.

In [ ]:
batch_size = 128

transform = transforms.Compose([transforms.ToTensor()])

train_dataset_full = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
train_dataset = torch.utils.data.Subset(train_dataset_full, range(10000))
mnist_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

print(f"Dataset MNIST carregado com {len(train_dataset)} exemplos")
print(f"Número de batches: {len(mnist_dataloader)}")

In [ ]:
def preprocess_images(x, alpha=1e-5):
    x = (x * 255.0 + torch.rand_like(x)) / 256.0
    x = alpha + (1 - 2 * alpha) * x
    x = torch.log(x) - torch.log(1 - x)
    return x

def postprocess_images(x):
    x = torch.sigmoid(x)
    return torch.clamp(x, 0, 1)

### Otimização e Treinamento (MNIST)

Definiremos nosso modelo com `input_dim = 784`, `hidden_dim = 512` e `num_layers = 6`. Treinaremos a rede por `10` épocas (o que leva cerca de 20-30 segundos em GPUs e menos de 2 minutos em CPUs normais) e salvaremos a curva de Negative Log-Likelihood.

In [ ]:
input_dim_mnist = 784
hidden_dim_mnist = 512
num_layers_mnist = 6

model_mnist = NormalizingFlow(input_dim_mnist, hidden_dim_mnist, num_layers_mnist).to(device)

In [ ]:
optimizer_mnist = optim.Adam(model_mnist.parameters(), lr=1e-3)
base_dist_mnist = torch.distributions.Normal(torch.zeros(input_dim_mnist).to(device), torch.ones(input_dim_mnist).to(device))

In [ ]:
num_epochs_mnist = 30
losses_mnist = []

for epoch in range(1, num_epochs_mnist + 1):
    model_mnist.train()
    epoch_loss = 0

    for real_imgs, _ in mnist_dataloader:
        real_imgs = real_imgs.view(-1, input_dim_mnist).to(device)
        real_imgs = preprocess_images(real_imgs)

        optimizer_mnist.zero_grad()

        z, log_det_jacobian = model_mnist(real_imgs)

        log_prob_z = base_dist_mnist.log_prob(z).sum(dim=1)
        log_prob_x = log_prob_z + log_det_jacobian

        loss = -log_prob_x.mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_mnist.parameters(), max_norm=5.0)
        optimizer_mnist.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(mnist_dataloader)
    losses_mnist.append(avg_loss)

    print(f"[Época {epoch}/{num_epochs_mnist}] Loss (NLL): {avg_loss:.4f}")

### Geração de Imagens

Diferente de VAEs e GANs, para gerar imagens de dígitos em Normalizing Flows basta amostrar vetores $z$ puramente Gaussianos a partir da distribuição base $\mathcal{N}(0, I)$ e mapeá-los de volta para o espaço de dados utilizando o fluxo invertível `model.inverse(z)`.

In [ ]:
model_mnist.eval()

with torch.no_grad():
    z_sample = base_dist_mnist.sample((16,))
    generated_imgs = model_mnist.inverse(z_sample)

    generated_imgs = postprocess_images(generated_imgs)
    generated_imgs = generated_imgs.view(-1, 1, 28, 28).cpu()

    fig, axs = plt.subplots(4, 4, figsize=(6, 6))

    for i, ax in enumerate(axs.flat):
        ax.imshow(generated_imgs[i].squeeze(), cmap="gray")
        ax.axis("off")

    plt.suptitle("Imagens Geradas pelo Normalizing Flow")
    plt.show()

## Exercícios

### Exercício 1
Modifique o número de camadas (`num_layers_2d`) do Normalizing Flow 2D para `2` e, em seguida, para `16`. Observe visualmente a diferença nos scatter plots gerados. O que acontece com a expressividade da rede quando usamos poucas camadas em um espaço bidimensional?

### Exercício 2
Experimente treinar o fluxo 2D em uma distribuição diferente. Utilize o `sklearn.datasets.make_circles(n_samples=3000, factor=0.5, noise=0.05, random_state=42)` para criar uma distribuição de dois anéis concêntricos. Treine o fluxo e plote a geração final, avaliando a suavidade da transformação encontrada pelas camadas de acoplamento.